# Deep dive 1: Embeddings & evaluation

*Rulebook notebooks: <a href="https://ecoop.github.io/rulebook/walkthrough.html" target="_parent">Start here: RAG walkthrough</a> · **Deep dive 1***

<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/embeddings_and_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Three ways to use this notebook**
> - **Read it, with results:** the <a href="https://ecoop.github.io/rulebook/embeddings_and_eval.html" target="_parent">results page</a> shows it after a fresh run.
> - **Run it in your browser:** <a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/embeddings_and_eval.ipynb" target="_parent">open it in Colab</a> (free; needs a Google account).
>   Already in Colab? Run the cells from the top.
> - **Run it on your machine:** see below.
>
> GitHub's own notebook view shows the code only: no results, and nothing runs there.

The <a href="https://ecoop.github.io/rulebook/walkthrough.html" target="_parent">RAG walkthrough</a> takes one quick look at each stage of the
pipeline. This deep dive slows down on the middle stage, embedding and retrieval, in two parts:

1. **What an embedding is**: text turned into a vector so similar meanings sit near each other.
2. **How to tell whether retrieval is any good**: where eyeballing stops working, and how a
   gold set plus `hit@k` / `MRR` let you compare embedders with numbers.

You don't need to have run the walkthrough first. This notebook runs **key-free**: everything
uses small local models, and nothing here calls generation, so nothing is billed.

To run it on your own machine, launch from the repo root: `uv run jupyter lab`. On Colab, the
first code cell fetches and installs the repo for you (about a minute).

In [ ]:
# On Colab, fetch the repo first (about a minute). On your own machine this is skipped.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/ecoop/rulebook /content/rulebook
    %pip install -q -e /content/rulebook
    sys.path.insert(0, "/content/rulebook/src")  # this session can't see the new install otherwise

from rulebook.notebook import setup, show_table
setup()

## 1. What an embedding is

The walkthrough used embeddings without opening one up. Here's what's inside.

An embedding turns text into a **vector** — a fixed-length list of numbers, the same length for
every text, short or long — chosen so that **similar meanings land near each other**. No single
number means anything on its own; what carries meaning is the *direction* the whole list points,
so the *same* model must embed both the stored rules and your question or the geometry wouldn't
line up. Below, embed **two pairs of near-paraphrases** — two ways of saying one checkers idea,
two ways of saying one cornhole idea — print one vector (truncated) to see what they actually
are, then look at how close they land (cosine similarity; 1.0 = same direction).

In [ ]:
import numpy as np
from rulebook.embeddings import get_embedder

emb = get_embedder()  # the default local model — no key, no call to a hosted provider
phrases = [
    "winning a game of checkers",                 # p0 ─┐ same idea,
    "how you win at checkers",                    # p1 ─┘ said two ways
    "tossing a beanbag onto the cornhole board",  # p2 ─┐ same idea,
    "throwing a bag in cornhole",                 # p3 ─┘ said two ways
]
vecs = np.asarray(emb.embed(phrases, input_type="document"))

print(f"each embedding is a fixed-length vector of {vecs.shape[1]} numbers")
print(f"p0, first 8 of {vecs.shape[1]}:  {np.round(vecs[0][:8], 3)} ...")
print(f"its length (norm) is {np.linalg.norm(vecs[0]):.2f}  "
      f"(unit vectors, so the dot products below ARE cosine similarities)")

sims = vecs @ vecs.T
labels = [f"p{i}" for i in range(len(phrases))]
show_table(
    [""] + labels + ["phrase"],
    [[lab] + [f"{v:.2f}" for v in row] + [p] for lab, row, p in zip(labels, sims, phrases)],
)

Each phrase is nearest its paraphrase partner — `p0↔p1` and `p2↔p3` are the biggest
off-diagonal numbers, and similarity drops across the two topics: *meaning*, not matching words,
decides nearness. It's a blunt signal, though — this is a lightweight static model — and you'll
see it bite in §4.

## 2. A tiny index to search

Retrieval needs an index: every rule chunked and embedded. Build the bundled checkers sample if
it isn't already — this is exactly what the quickstart's `build_index.py` does, key-free.

In [ ]:
from rulebook.notebook import ensure_index

ensure_index("checkers")  # runs scripts/build_index.py if the index is missing

## 3. Retrieve: an obvious winner

Given a question, embed it the same way and pull the nearest rules.
**Distance** is 2 − 2 × the cosine similarity of the two vectors: 0 means identical,
2 unrelated, 4 opposite. Lower is nearer.

In [ ]:
from rulebook.retrieve import retrieve


def show_top(question, k=3):
    """The top-k rules for a question, as a table."""
    rows = [
        (rank, c.rule_id, f"{c.distance:.2f}", c.text[:80] + "…")
        for rank, c in enumerate(retrieve(question, domain="checkers", k=k), start=1)
    ]
    show_table(["rank", "rule", "distance", "text"], rows)


show_top("If I touch one of my own pieces, must I move it?")

The touch-and-move rule comes first, well ahead of the runner-up — and you can tell it's the
right answer just by reading it. For a question like this, eyeballing works.

## 4. Retrieve: a muddled one

Now a question whose answer is just as plainly in the rules — *Object of the game* and *Winning*:

In [ ]:
show_top("How do you win a game of checkers?")

Look at what came first: the **preamble** — the file's title and introduction, which sit before
rule 1.1. It's *about* checkers games, so it lands close to the question, but it never says how
to win. The rules that do answer the question are further down. Hand only the top result to an
LLM and it gets the wrong context.

That's a **chunking** effect as much as a model one: the chunker cuts the file at each numbered
rule, and everything before the first rule becomes its own "preamble" chunk. Changing how
documents are cut up is one fix. Here, let's try changing the model instead.

## 5. Change one thing: the embedder

An **embedder** is the model that turns text into the vector from §1. Swap it and you change the
geometry — so the rankings change.

The three local embedders below are **model2vec "potion"** models from
[MinishLab](https://huggingface.co/minishlab) on Hugging Face: small *static* models distilled
from larger embedding models, downloaded once and run on your CPU. They differ mainly in size
(2M / 8M / 32M parameters), and one is tuned specifically for retrieval. The app also supports two
hosted embedders, **Voyage** and **OpenAI**, and many other providers exist.

| | Local (model2vec) | Hosted (Voyage, OpenAI) |
|---|---|---|
| Cost | free | per token (small) |
| Setup | none | API key + account |
| Privacy | text never leaves your machine | text is sent to the provider |
| Quality | small, static (ignores word context) | larger, contextual |
| Offline | yes | no |

Below, the same two questions go through all three. Each model gets its own temporary index, so
your real index isn't touched. (The first run downloads two more small models.)

In [ ]:
from rulebook.config import settings
from rulebook.eval import build_scratch_index, rank_of

MODELS = [
    "minishlab/potion-retrieval-32M",
    "minishlab/potion-base-8M",
    "minishlab/potion-base-2M",
]
indexes = {m: build_scratch_index("checkers", m) for m in MODELS}


def label(model):
    name = model.split("/")[-1]
    return f"{name} (default)" if model == settings.embedding_model else name


def where(question, citations, index):
    r = rank_of(question, citations, index=index)
    return "not in top 5" if r is None else ("**#1**" if r == 1 else f"#{r}")


QUESTIONS = [
    ("If I touch one of my own pieces, must I move it?", ("checkers 5.3",)),
    ("How do you win a game of checkers?", ("checkers 5.1", "checkers 2.1")),
]
show_table(
    ["question", "citations"] + [label(m) for m in MODELS],
    [[q, " / ".join(c)] + [where(q, c, indexes[m]) for m in MODELS] for q, c in QUESTIONS],
)

The first question was never in doubt. The second one depends entirely on the embedder — compare
where the answer lands in each column.

One thing that *didn't* change: **`k`**, how many passages retrieval returns. `k` never reorders
results; it only decides how many the LLM gets to read. If a model puts the answer at #3, then
`k=1` hands the LLM the wrong passage and `k=3` only just includes the right one.

So should you switch models — on the strength of **two** questions? You'd want to check many
more. And the next time you change something (the model, the chunking, `k`), you'd want to check
them all again, the same way.

## 6. Write the answers down once: golds

When you looked at §3 and saw that touch-and-move was right, you were checking against an answer
you already knew. A **gold** is that knowledge written down: a question paired with its right
answer. Here the answer is given as **citations** — the places in the rules that answer it,
written the way the app cites them (`checkers 5.3`) — so these are **citation golds**. Write a
set of them once, and you can score *any* retrieval setup against it — automatically, and the
same way every time.

Here's a minimal gold set for checkers. Each line of `eval/gold/checkers.jsonl` is one gold:

```json
{"domain": "checkers", "question": "How do you win a game of checkers?", "citations": ["checkers 5.1", "checkers 2.1"], "style": "literal"}
```

In [ ]:
from rulebook.eval import load_gold

gold = load_gold("checkers")
show_table(["question", "citations"], [(g.question, " / ".join(g.citations)) for g in gold])

Some golds cite two rules. *How do you win?* is answered by both **2.1 Object of the game** and
**5.1 Winning**, which say the same thing in different words. Retrieving either one counts as
finding the answer, so a model isn't marked wrong for picking the other right rule.

Two standard scores turn "did retrieval find it?" into numbers:

- **hit@k** — the share of questions with a cited rule in the top *k*. *Is the answer
  even in what the LLM gets to read?*
- **MRR** (mean reciprocal rank) — score each question 1 / (rank of the first cited rule): #1 → 1.0,
  #2 → 0.5, #3 → 0.33, missing → 0; then average. It rewards putting the answer *first*, not just
  somewhere.

The same `evaluate()` runs here and behind `python -m rulebook.eval`, so the notebook and the app
report the same numbers.

In [ ]:
from rulebook.eval import evaluate

rows = []
for m in MODELS:
    res = evaluate(gold, k=5, index=indexes[m])
    rows.append([label(m)] + [f"{res.hit_rate_at(k):.2f}" for k in (1, 3, 5)] + [f"{res.mrr:.2f}"])
show_table(["model", "hit@1", "hit@3", "hit@5", "MRR"], rows)

What two questions couldn't tell you:

- **"Better" isn't one number.** The smallest model puts a cited rule *first* more often than
  the default (higher hit@1), yet finds one in the top 5 no more often (same hit@5). Which
  column matters depends on `k`: this app hands the LLM 5 passages, so a rule at #4 is fine and
  a rule that's missing entirely is not.
- **Bigger, or "tuned for retrieval," isn't automatically better.** Check where the
  retrieval-tuned model lands before assuming it wins.
- **Mind the sample size.** With 12 questions, one question moves hit@k by about 0.08. A gold set
  doesn't have to be big to be useful, but it's evidence, not proof.

## 7. Local vs. a hosted embedder (Voyage)

The same gold set answers *"is a hosted model worth it?"* with a number. Hosted models aren't
local, so this one uses the real index (needs a `VOYAGE_API_KEY`, which costs a little):

```bash
# 1. rebuild the checkers index with Voyage vectors
EMBEDDING_PROVIDER=voyage EMBEDDING_MODEL=voyage-4 \
    uv run python scripts/build_index.py --domain checkers
# 2. score it against the same gold set
EMBEDDING_PROVIDER=voyage EMBEDDING_MODEL=voyage-4 \
    uv run python -m rulebook.eval --domain checkers
# 3. put the default back when you're done
uv run python scripts/build_index.py --domain checkers
```

Compare its scores with the table above. On a small, clean corpus like this, a hosted model may
not win by much — and that's a useful result too. Either way it's evidence instead of a guess.

## 8. The other kind of gold: written golds

Everything above used **citation golds**: the answer points into the rules. The full app (the
web UI) also has **written golds**: answers written out as text by a signed-in user, usually to
fix or extend an answer the app gave. They can cover what the rules don't — *"I have 10 players:
ultimate or goaltimate?"* or *"What does it mean to play 'savage'?"*

| | citation gold | written gold |
|---|---|---|
| The answer is | places in the rules (`checkers 5.1`) | text |
| Written by | whoever builds the eval | users, in the app |
| Stored in | `eval/gold/<domain>.jsonl` | the app's gold log, curated by an admin |
| Used for | scoring retrieval (this notebook) | answering: indexed next to the rules, so similar questions retrieve it |

They're the same idea — a question with its right answer — and
[#254](https://github.com/ecoop/rulebook/issues/254) brings them into one shape, so a written
gold can carry citations too. One thing to keep in mind when you get there: a written gold's
own text is always the nearest match to its own question, so retrieval eval scores against the
rules, not against written golds.

## Takeaway

- An embedding is a fixed-length vector; retrieval finds the vectors nearest the question.
- One question at a time, you can judge results by eye — until the answer is buried, or you want
  to compare two setups.
- A **gold** is a question with its right answer written down once. With **citation golds**,
  **hit@k** and **MRR** score any setup against them, so "is this change better?" gets a number.
- Measure before you upgrade: the bigger or fancier model isn't always the better one.
- The app's **written golds** are the same idea with a text answer; they help answer questions
  rather than score retrieval.